# Ablation Significance Analysis

Checks the three ablation studies (`ablations/ablation1_model_comparison.py`,
`ablation2_curve_preprocessing.py`, `ablation3_generalisability.py`) for statistically
significant differences between conditions, reusing the exact same Friedman + pairwise
Wilcoxon (Holm-Bonferroni corrected) / McNemar framework as `08_statistical_comparison.py`
-- no test logic is reimplemented here, just adapted to the ablation scripts' flat
per-dataset `*_performances.joblib` schema (same per-model keys as
`classification_performances.joblib`, just one dataset's results per file instead of
nested under `{dataset_name: {mode: {...}}}`).

**Primary test (N >= 2 datasets):** Friedman test across datasets (one mean-accuracy
value per dataset per condition -> independent observations), then pairwise Wilcoxon
signed-rank with Holm-Bonferroni correction, effect size = rank-biserial correlation.

**Fallback (N = 1 dataset):** McNemar's test on concatenated per-sample outcomes.


In [ ]:
import os, sys, importlib.util
import joblib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch, FancyArrowPatch
from pathlib import Path

# Run from the notebook's own directory's parent (main/), matching house style.
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(os.path.dirname(os.path.dirname(_nb))))
except Exception:
    pass

sys.path.insert(0, 'utils')
%load_ext autoreload
%autoreload 2
import config
import statistical_comparison as stat_comparison

# stat_comparison imports matplotlib and calls matplotlib.use("Agg") at module load --
# re-assert the inline backend afterwards so figures render in this notebook.
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linestyle': '--',
    'grid.alpha': 0.7,
    'font.size': 10,
})
print("CWD:", os.getcwd())


In [ ]:
from IPython.display import HTML, display

# Same 5 LAB_DDM_paper folders the user's ablation-1/3 SLURM array jobs target
# (01, 02, 03, 09, 10 -- see abl_ablation1_model_comparison.sh / abl_ablation3_generalisability.sh).
LAB_DATASETS = ['01_ACA_qdPCR', '02_AMCA_qdLAMP', '03_AMCA_qdPCR']

# Mirrors each ablation script's MODELS list exactly -- keep these in sync if the
# scripts' lists change.
ABLATION1_MODELS = ['knn', 'cnn', 'gru', 'transformer', 'cnn_gru_dual', 'cnn_gru_dual_attn_recon']
ABLATION3_CGD_FAMILY = ['cnn_gru_dual', 'cnn_gru_dual_mtl', 'gru_rcfd_cgd', 'cnn_gru_dual_supcon3']
ABLATION3_ATTN_FAMILY = ['cnn_gru_dual_attn_recon', 'cnn_gru_dual_attn_recon_mtl',
                         'gru_rcfd_attn_recon', 'cnn_gru_dual_attn_recon_supcon3']

ALPHA = 0.05
METRIC = 'accuracy'

def show_result(stats, figs, test_type, metric_name='Accuracy', alpha=ALPHA, title=None):
    """Render one comparison's summary table + figures inline, reusing
    stat_comparison's HTML/plot builders untouched."""
    if title:
        display(HTML(f'<h3>{title}</h3>'))
    if not stats or 'error' in (stats or {}):
        display(HTML(f'<p style="color:#e74c3c">{(stats or {}).get("error", "No data.")}</p>'))
        return
    html = stat_comparison.build_tab_content(stats, figs, test_type, metric_name, alpha)
    display(HTML(html))
    for fig in figs.values():
        display(fig)
        plt.close(fig)


In [ ]:
def load_ablation_exp_data(joblib_path, models, curve_type='ori_curve', metric=METRIC):
    """Adapt one ablation *_performances.joblib (flat {filter: {...}} schema --
    same per-model y_preds_AC_*_/y_trues_ keys as classification_performances.joblib,
    just one dataset's results, not nested under {dataset_name: {mode: {...}}}) into
    the {curve_type: {filter: {model: {fold_metrics, mean_metric, y_true_all,
    y_pred_all, n_folds}}}} shape stat_comparison.build_accuracy_matrix/_get_value
    expect. Mirrors stat_comparison.load_exp_data's inner loop exactly."""
    joblib_path = Path(joblib_path)
    if not joblib_path.exists():
        return None
    raw = joblib.load(joblib_path)

    out_filters = {}
    for filter_key, filter_results in raw.items():
        if "y_trues_" not in filter_results:
            continue
        y_trues_ = filter_results["y_trues_"]
        n_folds = len(y_trues_)
        if n_folds == 0:
            continue

        out_filters[filter_key] = {}
        for model_key in models:
            preds_key = config.MODEL_KEY_MAP[model_key][0]
            if preds_key not in filter_results:
                continue
            y_preds_ = filter_results[preds_key]
            if len(y_preds_) != n_folds:
                continue

            fold_metrics, y_true_parts, y_pred_parts = [], [], []
            for fi in range(n_folds):
                y_true = np.asarray(y_trues_[fi])
                y_pred = np.asarray(y_preds_[fi])
                if len(y_true) == 0 or len(y_true) != len(y_pred):
                    continue
                fold_metrics.append(stat_comparison._compute_metric(y_true, y_pred, metric))
                y_true_parts.append(y_true)
                y_pred_parts.append(y_pred)

            if not fold_metrics:
                continue
            out_filters[filter_key][model_key] = {
                "fold_metrics": fold_metrics,
                "mean_metric":  float(np.mean(fold_metrics)),
                "y_true_all":   np.concatenate(y_true_parts),
                "y_pred_all":   np.concatenate(y_pred_parts),
                "n_folds":      len(fold_metrics),
            }
    return {curve_type: out_filters}


def load_ablation_across_datasets(exp_folder, dataset_names, joblib_name, models,
                                  curve_type='ori_curve', metric=METRIC):
    """One (name, exp_data) pair per dataset with usable results -- the shape
    stat_comparison.run_comparison's exp_data_list expects (one entry per
    independent 'experiment' for the Friedman test)."""
    exp_data_list = []
    for name in dataset_names:
        path = Path(exp_folder) / name / "ablations" / joblib_name
        data = load_ablation_exp_data(path, models, curve_type=curve_type, metric=metric)
        if data is not None and any(data[curve_type].values()):
            exp_data_list.append((name, data))
        else:
            print(f'  [WARN] no usable results for {name} at {path}')
    return exp_data_list


## Ablation 1 -- Model Comparison

kNN vs CNN vs GRU vs Transformer vs cnn_gru_dual vs cnn_gru_dual_attn_recon, across the 5 LAB_DDM_paper datasets. `cnn_gru_dual_attn_recon` is soft-skipped on LAB data (no pixel-grid metadata) -- if it has zero results across all 5 datasets it's automatically dropped from the comparison by `build_accuracy_matrix` (conditions with any missing data are excluded).

In [ ]:
exp_data_1 = load_ablation_across_datasets(
    config.LAB_EXP_FOLDER, LAB_DATASETS,
    "ablation1_model_comparison_performances.joblib", ABLATION1_MODELS)

stats1, figs1, test_type1, _ = stat_comparison.run_comparison(
    exp_data_1,
    compare_axis="models",
    all_conditions=ABLATION1_MODELS,
    condition_print_map=config.MODEL_PRINT_MAP,
    fixed={"curve_type": "ori_curve", "filter": None},
    baseline_key="cnn_gru_dual",
    metric=METRIC,
    alpha=ALPHA,
)
show_result(stats1, figs1, test_type1, title="Ablation 1: Model Comparison")


In [ ]:
config.MODEL_PRINT_MAP

In [ ]:
DATASET_LABELS_1 = {
    '01_ACA_qdPCR':   'qdPCR (3-plex)\n(Moniri et al., 2020a)',
    '03_AMCA_qdPCR':  'qdPCR (9-plex)\n(Moniri et al., 2020b)',
    '02_AMCA_qdLAMP': 'qdLAMP (5-plex)\n(Malpartida-Cardenas, et al., 2022)',
}
DATASET_ORDER_1 = ['01_ACA_qdPCR', '03_AMCA_qdPCR', '02_AMCA_qdLAMP']
BASELINE_MODELS_1 = {'01_ACA_qdPCR': 'knn', '03_AMCA_qdPCR': 'cnn', '02_AMCA_qdLAMP': 'knn'}

MODEL_COLORS_1 = {
    'knn':                     '#85C1E9',
    'cnn':                     '#82E0AA',
    'gru':                     '#F5B041',
    'transformer':             '#EC7063',
    'cnn_gru_dual':            '#BB8FCE',
    'cnn_gru_dual_attn_recon': '#F7DC6F',
}

MODEL_PRINT_MAP = {
    'knn':                     'KNN',
    'cnn':                     'CNN',
    'gru':                     'BiGRU',
    'transformer':             'Transformer',
    'cnn_gru_dual':            'CNN-BiGRU Dual Branch',
    'cnn_gru_dual_attn_recon': 'CNN-BiGRU Dual + Attn Recon',
}

def plot_ablation1_barchart(exp_data, models, baseline_map, model_colors, dataset_order=None,
                            dataset_labels=None, curve_type='ori_curve', filter_key=None):
    data_by_name = dict(exp_data)
    dataset_names = [n for n in (dataset_order or data_by_name) if n in data_by_name]
    present_models = [m for m in models
                       if any(m in data_by_name[n][curve_type].get(filter_key, {}) for n in dataset_names)]

    acc = np.full((len(dataset_names), len(present_models)), np.nan)
    for i, name in enumerate(dataset_names):
        entry = data_by_name[name][curve_type].get(filter_key, {})
        for j, m in enumerate(present_models):
            if m in entry:
                acc[i, j] = entry[m]['mean_metric'] * 100
    best_j = np.nanargmax(acc, axis=1)

    n_models = len(present_models)
    width = 0.8 / n_models
    x = np.arange(len(dataset_names))

    fig, ax = plt.subplots(figsize=(2 * len(dataset_names) + 2, 4))
    for j, m in enumerate(present_models):
        offsets = x + (j - (n_models - 1) / 2) * width
        bars = ax.bar(offsets, acc[:, j], width=width, color=model_colors.get(m, '#95A5A6'),
                      label=MODEL_PRINT_MAP.get(m, m), zorder=3)
        for i, bar in enumerate(bars):
            v = acc[i, j]
            if np.isnan(v):
                continue
            is_baseline = baseline_map.get(dataset_names[i]) == m
            is_best = j == best_j[i]
            if is_baseline:
                bar.set_linestyle('--')
                bar.set_edgecolor('#555555')
                bar.set_linewidth(1.3)
            if is_best:
                ax.text(bar.get_x() + bar.get_width() / 2, v + 2.6, '\u2605',
                        ha='center', va='bottom', fontsize=11, color='#B7950B', zorder=4)
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.6, f'{v:.2f}',
                    ha='center', va='bottom', fontsize=8,
                    fontweight='bold' if is_best else 'normal')

    ax.set_xticks(x)
    ax.set_xticklabels([(dataset_labels or {}).get(n, n) for n in dataset_names])
    # ax.set_xlabel('Dataset')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 112)
    fig.suptitle('Phase 1: Accuracy by Dataset and Model', fontsize=13, y=1.03)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=n_models, frameon=False)
    fig.tight_layout()
    return fig


fig_bar1 = plot_ablation1_barchart(
    exp_data_1, ABLATION1_MODELS, BASELINE_MODELS_1, MODEL_COLORS_1,
    dataset_order=DATASET_ORDER_1, dataset_labels=DATASET_LABELS_1,
)
plt.show()


## Ablation 1 (cont.) -- Full Metrics Table

Accuracy, macro-averaged F1/precision/sensitivity/specificity, MCC, and per-target
sensitivity/specificity, for every (dataset, model) pair -- one table per dataset since
each LAB dataset has a different target panel (3/9/5-plex).

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder


def get_class_names(exp_path):
    exp_path = Path(exp_path)
    training_data = joblib.load(exp_path / config.TRAINING_DATA_PATH)
    Y_well = training_data["Y_well"]
    label_mappings = config.get_label_mappings(exp_path)
    if exp_path.name in label_mappings:
        mapping = label_mappings[exp_path.name]
        Y_well = [mapping.get(w, w) for w in Y_well]
    return LabelEncoder().fit(Y_well).classes_


def compute_full_metrics(y_true, y_pred, class_names):
    labels = np.arange(len(class_names))
    report = classification_report(y_true, y_pred, labels=labels, target_names=list(class_names),
                                   output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    row = {
        "n_test":      len(y_true),
        "accuracy":    accuracy_score(y_true, y_pred),
        "precision":   report["macro avg"]["precision"],
        "sensitivity": report["macro avg"]["recall"],
        "f1":          report["macro avg"]["f1-score"],
        "mcc":         stat_comparison._compute_metric(y_true, y_pred, "mcc"),
    }
    specs = []
    for i, cls in enumerate(class_names):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - cm[i, :].sum() - fp
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        specs.append(spec)
        row[f"sensitivity__{cls}"] = report[cls]["recall"]
        row[f"specificity__{cls}"] = spec
    row["specificity"] = float(np.nanmean(specs))
    return row


metrics_rows = []
for dataset_name, exp_data in exp_data_1:
    class_names = get_class_names(Path(config.LAB_EXP_FOLDER) / dataset_name)
    entry = exp_data["ori_curve"].get(None, {})
    for model_key in ABLATION1_MODELS:
        if model_key not in entry:
            continue
        d = entry[model_key]
        m = compute_full_metrics(d["y_true_all"], d["y_pred_all"], class_names)
        metrics_rows.append(dict(dataset=dataset_name, model=model_key, **m))

metrics_df = pd.DataFrame(metrics_rows)
print(f"Built metrics for {len(metrics_df)} (dataset, model) pairs.")
metrics_df[["dataset", "model", "n_test", "accuracy", "f1", "sensitivity", "specificity", "mcc"]]

In [ ]:
model_order = [m for m in ABLATION1_MODELS if m in metrics_df.model.unique()]

for dataset_name, _ in exp_data_1:
    class_names = get_class_names(Path(config.LAB_EXP_FOLDER) / dataset_name)
    sub = metrics_df[metrics_df.dataset == dataset_name].set_index("model").reindex(model_order).dropna(how="all")

    data = {
        ("Overall", "N"):                  sub["n_test"],
        ("Overall", "Accuracy"):           sub["accuracy"],
        ("Overall", "F1 (macro)"):         sub["f1"],
        ("Overall", "Precision (macro)"):  sub["precision"],
        ("Overall", "MCC"):                sub["mcc"],
    }
    for c in class_names:
        data[("Sensitivity", c)] = sub[f"sensitivity__{c}"]
    data[("Sensitivity", "Macro")] = sub["sensitivity"]
    for c in class_names:
        data[("Specificity", c)] = sub[f"specificity__{c}"]
    data[("Specificity", "Macro")] = sub["specificity"]

    table = pd.DataFrame(data)
    table.index = [config.MODEL_PRINT_MAP.get(m, m) for m in table.index]

    pct_cols = [c for c in table.columns if c[1] not in ("N", "MCC")]
    best_cols = pct_cols + [("Overall", "MCC")]  # higher-is-better columns; N excluded
    display_table = table.copy()
    display_table[pct_cols] = display_table[pct_cols] * 100

    fmt = {c: "{:.2f}%" for c in pct_cols}
    fmt[("Overall", "N")] = "{:.0f}"
    fmt[("Overall", "MCC")] = "{:.3f}"

    styled = (display_table.style
             .format(fmt)
             .highlight_max(subset=best_cols, color="#a8e6a1")
             .set_caption(f"{dataset_name} -- Ablation 1 full metrics (row = model)"))
    display(styled)

In [ ]:
LATEX_MODEL_LABELS_T1 = {'knn': '$k$-NN', 'cnn': 'CNN', 'gru': 'BiGRU',
                         'transformer': 'Transformer', 'cnn_gru_dual': 'CNN-BiGRU Dual Branch'}
LATEX_BASELINE_1 = {'01_ACA_qdPCR': 'knn', '03_AMCA_qdPCR': 'cnn', '02_AMCA_qdLAMP': 'knn'}
LATEX_SUBCAPTION_T1 = {
    '01_ACA_qdPCR':  'Performance on the qdPCR 3-plex dataset from Moniri et al.~\\cite{paper203}',
    '03_AMCA_qdPCR': 'Performance on the qdPCR 9-plex dataset from Moniri et al.~\\cite{paper204}',
    '02_AMCA_qdLAMP': 'Performance on the qdLAMP 5-plex dataset from Malpartida-Cardenas et al.~\\cite{paper206}',
}
LATEX_ORDER_T1 = ['01_ACA_qdPCR', '03_AMCA_qdPCR', '02_AMCA_qdLAMP']
T1_METRICS_MAIN = ['accuracy', 'f1', 'sensitivity', 'specificity']


def load_fold_pairs(dataset_name, model_key):
    path = Path(config.LAB_EXP_FOLDER) / dataset_name / "ablations" / "ablation1_model_comparison_performances.joblib"
    raw = joblib.load(path)
    res = raw.get(None, {})
    preds_key = config.MODEL_KEY_MAP[model_key][0]
    if preds_key not in res or "y_trues_" not in res:
        return []
    pairs = []
    for yt, yp in zip(res["y_trues_"], res[preds_key]):
        yt, yp = np.asarray(yt), np.asarray(yp)
        if len(yt) and len(yt) == len(yp):
            pairs.append((yt, yp))
    return pairs


def compute_mean_std_metrics(dataset_name, model_key, class_names):
    """Mean +- std of each metric across CV folds (not the pooled/concatenated point estimate)."""
    pairs = load_fold_pairs(dataset_name, model_key)
    if not pairs:
        return None
    fold_rows = [compute_full_metrics(yt, yp, class_names) for yt, yp in pairs]
    out = {}
    for k in T1_METRICS_MAIN:
        vals = np.array([r[k] for r in fold_rows])
        out[f"{k}_mean"] = vals.mean()
        out[f"{k}_std"] = vals.std(ddof=1) if len(vals) > 1 else 0.0
    return out


def fmt_pct(v):
    return f"{v * 100:.2f}\\%"


def cell_t1(mean, std, is_best):
    v = f"{fmt_pct(mean)} $\\pm$ {std * 100:.2f}\\%"
    return f"\\textbf{{{v}}}" if is_best else v


def build_table1_subtable(dataset_name, letter):
    class_names = get_class_names(Path(config.LAB_EXP_FOLDER) / dataset_name)
    per_model = {m: compute_mean_std_metrics(dataset_name, m, class_names) for m in model_order}
    per_model = {m: v for m, v in per_model.items() if v is not None}
    best = {k: max(per_model, key=lambda m: per_model[m][f"{k}_mean"]) for k in T1_METRICS_MAIN}

    lines = [
        f"    ({letter}) {LATEX_SUBCAPTION_T1[dataset_name]}\\\\[0.5em]",
        "    \\resizebox{\\textwidth}{!}{",
        "    \\begin{tabular}{lrrrr}",
        "    \\toprule",
        "     & \\multicolumn{2}{c}{\\textbf{Overall}} & \\multicolumn{2}{c}{\\textbf{Macro Averages}} \\\\",
        "    \\cmidrule(lr){2-3} \\cmidrule(lr){4-5}",
        "    \\textbf{Model} & \\textbf{Accuracy} & \\textbf{F1} & \\textbf{Sensitivity} & \\textbf{Specificity} \\\\",
        "    \\midrule",
    ]
    for model_key in model_order:
        if model_key not in per_model:
            continue
        ms = per_model[model_key]
        label = LATEX_MODEL_LABELS_T1.get(model_key, model_key)
        if LATEX_BASELINE_1.get(dataset_name) == model_key:
            label += " \\textit{(Baseline)}"
        if model_key == "cnn_gru_dual":
            label = f"\\textbf{{{label}}}"
        cells = [cell_t1(ms[f"{k}_mean"], ms[f"{k}_std"], best[k] == model_key) for k in T1_METRICS_MAIN]
        lines.append(f"    {label} & " + " & ".join(cells) + " \\\\")
    lines += ["    \\bottomrule", "    \\end{tabular}", "    }"]
    return "\n".join(lines)


subtables_1 = [build_table1_subtable(name, letter) for name, letter in zip(LATEX_ORDER_T1, "abc")]
table1_latex = (
    "\\begin{table}[htbp]\n"
    "    \\centering\n"
    "    \\caption{Overall and macro-averaged performance metrics across the three evaluated datasets. "
    "Bold marks the best result on each dataset.}\n"
    "    \\label{tab:ablation1_overall_metrics}\n"
    "    \\small\n\n"
    + "\n\n    \\vspace{2.5em}\n\n".join(subtables_1)
    + "\n\\end{table}"
)
print(table1_latex)


In [ ]:
LATEX_MODEL_LABELS_T2 = {'knn': 'KNN', 'cnn': 'CNN', 'gru': 'BiGRU',
                         'transformer': 'Transformer', 'cnn_gru_dual': 'CNN-BiGRU Dual Branch'}
LATEX_SUBCAPTION_T2 = {
    '01_ACA_qdPCR':  'Performance on the qdPCR dataset from Moniri et al.~\\cite{paper203}',
    '02_AMCA_qdLAMP': 'Performance on the qdLAMP dataset from Malpartida-Cardenas et al.~\\cite{paper206}',
    '03_AMCA_qdPCR': 'Performance on the qdPCR dataset from Moniri et al.~\\cite{paper204}',
}
LATEX_ORDER_T2 = ['01_ACA_qdPCR', '02_AMCA_qdLAMP', '03_AMCA_qdPCR']


def compute_per_class_sens_spec_folds(dataset_name, model_key, class_names):
    pairs = load_fold_pairs(dataset_name, model_key)
    if not pairs:
        return None
    n_cls = len(class_names)
    fold_sens, fold_spec = [], []
    for yt, yp in pairs:
        cm = confusion_matrix(yt, yp, labels=range(n_cls))
        sens_c, spec_c = [], []
        for i in range(n_cls):
            tp = cm[i, i]
            fn = cm[i, :].sum() - tp
            fp = cm[:, i].sum() - tp
            tn = cm.sum() - tp - fn - fp
            sens_c.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
            spec_c.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
        fold_sens.append(sens_c)
        fold_spec.append(spec_c)
    fold_sens = np.array(fold_sens, dtype=float)
    fold_spec = np.array(fold_spec, dtype=float)
    return {
        'sens_mean': np.nanmean(fold_sens, axis=0) * 100, 'sens_std': np.nanstd(fold_sens, axis=0) * 100,
        'spec_mean': np.nanmean(fold_spec, axis=0) * 100, 'spec_std': np.nanstd(fold_spec, axis=0) * 100,
    }


def build_ablation1_per_class_subtable(dataset_name, letter):
    class_names = list(get_class_names(Path(config.LAB_EXP_FOLDER) / dataset_name))
    per_model = {}
    for m in model_order:
        stats = compute_per_class_sens_spec_folds(dataset_name, m, class_names)
        if stats is not None:
            per_model[m] = stats
    models_here = [m for m in model_order if m in per_model]

    model_header = ' & '.join(
        f'\\multicolumn{{2}}{{c}}{{\\textbf{{{LATEX_MODEL_LABELS_T2.get(m, m)}}}}}'
        for m in models_here)
    model_cmid = ' '.join(f'\\cmidrule(lr){{{2 + 2*i}-{3 + 2*i}}}' for i in range(len(models_here)))
    metric_row = ' & '.join('\\textbf{Sens.} & \\textbf{Spec.}' for _ in models_here)

    lines = [
        f"    ({letter}) {LATEX_SUBCAPTION_T2[dataset_name]}\\\\[0.5em]",
        "    \\resizebox{\\textwidth}{!}{",
        "    \\begin{tabular}{l" + "cc" * len(models_here) + "}",
        "    \\toprule",
        f"     & {model_header} \\\\",
        f"    {model_cmid}",
        f"    \\textbf{{Class}} & {metric_row} \\\\",
        "    \\midrule",
    ]
    for ci, cname in enumerate(class_names):
        sens_vals = {m: per_model[m]['sens_mean'][ci] for m in models_here}
        spec_vals = {m: per_model[m]['spec_mean'][ci] for m in models_here}
        best_sens = max(sens_vals, key=sens_vals.get) if sens_vals else None
        best_spec = max(spec_vals, key=spec_vals.get) if spec_vals else None

        cells = []
        for m in models_here:
            s = per_model[m]
            sens_str = f"{s['sens_mean'][ci]:.2f}\\% $\\pm$ {s['sens_std'][ci]:.2f}\\%"
            spec_str = f"{s['spec_mean'][ci]:.2f}\\% $\\pm$ {s['spec_std'][ci]:.2f}\\%"
            if m == best_sens:
                sens_str = f'\\textbf{{{sens_str}}}'
            if m == best_spec:
                spec_str = f'\\textbf{{{spec_str}}}'
            cells.append(sens_str)
            cells.append(spec_str)
        lines.append(f"    {cname} & " + " & ".join(cells) + " \\\\")
    lines += ["    \\bottomrule", "    \\end{tabular}", "    }"]
    return "\n".join(lines)


subtables_2 = [build_ablation1_per_class_subtable(name, letter) for name, letter in zip(LATEX_ORDER_T2, "abc")]
table2_latex = (
    "\\begin{landscape}\n"
    "\\begin{table}[!htbp]\n"
    "    \\centering\n"
    "    \\caption{Per-class sensitivity and specificity across five-fold cross-validation, per dataset. "
    "Results are mean $\\pm$ standard deviation across folds. Bold marks the best result per class.}\n"
    "    \\label{tab:ablation1_perclass_metrics}\n"
    "    \\small\n\n"
    + "\n\n    \\vspace{2.5em}\n\n".join(subtables_2)
    + "\n\\end{table}\n"
    "\\end{landscape}"
)
print(table2_latex)
